# Lab 5: ระบบ Vectorless RAG Part I

Vectorless RAG คือแนวทาง RAG ที่ **ไม่ใช้ Dense Embeddings และ Vector Similarity Search** ในการค้นข้อมูล

ระบบอาจค้นข้อมูลด้วยวิธีอื่น เช่น:

- Metadata Filtering
- SQL หรือ Database Query
- Keyword Search หรือ Inverted Index
- Hierarchical Navigation จาก Category/Section
- LLM Reasoning เพื่อเลือกเอกสารจาก Candidate Set

Vectorless ไม่ได้หมายความว่า “ไม่มี Retrieval” แต่หมายถึง Retrieval ไม่ได้อาศัย Vector Space เป็นกลไกหลัก



## เป้าหมาย: 

พัฒนาสร้างระบบแนะนำกระทู้ Pantip ที่เกี่ยวข้อง โดยอ้างอิงข้อมูล [krathu-500
](https://github.com/Pittawat2542/krathu-500) 

โดยอาศัยโครงสร้างข้อมูล (metadata/hierarchy) ที่มีอยู่แล้ว ร่วมกับ LLM ในการ Hierarchical Navigation แทนการค้นหาด้วยความคล้ายของเวกเตอร์

โดยจะแบ่งเป็น 2 Parts ได้แก่

#### Part I: Indexing 

1. **Analyse** — จัดกลุ่มข้อมูลตามหมวดหมู่ที่มีอยู่แล้วในข้อมูล (metadata) แทนการทำ embedding
2. **Structured Index** - แปลงข้อมูลเป็นคลังความรู้แบบ wiki ตาม Open Knowledge Format (OKF) 

#### Part II: Generating 

1. **Query Routing** — สร้าง agent ที่ค้นข้อมูลด้วยการไล่ตาม index/link 
2. **Generation with Sourcing** — ให้ LLM ตอบพร้อมอ้างอิงหมายเลขแหล่งที่มา



## ภาพรวมสถาปัตยกรรม

หลักการ คือ Retrieval Agent อ่านเฉพาะ Markdown Index ของ Page ปัจจุบันและรายการลิงก์ที่ parse ได้ แล้วต้องเลือกหนึ่งลิงก์จากรายการนี้เท่านั้น


```mermaid
flowchart TB
  OKF["OKF wiki bundle"] --> R1["Retrieval Agent"]
  style OKF fill:#ff9999,stroke:#333
      
  Q["Question"] --> R1
  style Q fill:#ff9999,stroke:#333
  R1 --> A["Answer + citations"]

```

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
MODEL_NAME = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(api_key=GEMINI_KEY, model=MODEL_NAME)

In [2]:
from pathlib import Path

ROOT_DIR = "./pantip-wiki"
root = Path(ROOT_DIR)

## Pipeline 1 — Simple Iterative Retrieval

Agent เริ่มที่ root `index.md` และทำซ้ำ `read → decide` บนเส้นทางเดียว:

```mermaid
flowchart TB
  A["initialize at index.md"] --> B["read_page"]
  B --> C["decide"]
  C -->|follow one link| B
  C -->|enough evidence| D["answer"]
  C -->|dead end / budget| E["abstain"]
```


In [3]:
import json
import re
from urllib.parse import urlparse
from pydantic import BaseModel, Field
import yaml

def split_frontmatter(text: str) -> tuple[dict, str]:
    if not text.startswith("---\n"):
        return {}, text
    _, frontmatter, body = text.split("---", 2)
    return yaml.safe_load(frontmatter) or {}, body.lstrip()


MARKDOWN_LINK_RE = re.compile(r"\[([^\]]+)\]\(([^)]+)\)")


def local_markdown_targets(page: Path, body: str, root: Path) -> list[Path]:
    targets = []
    for _, href in MARKDOWN_LINK_RE.findall(body):
        parsed = urlparse(href)
        if parsed.scheme or href.startswith("#"):
            continue
        clean = parsed.path
        target = (page.parent / clean).resolve()
        if target.is_dir():
            target = target / "index.md"
        targets.append(target)
    return targets
    

In [5]:
class PageLink(BaseModel):
    label: str
    href: str
    target_path: str


class OKFPage(BaseModel):
    path: str
    metadata: dict
    body: str
    links: list[PageLink]


def safe_resolve(root: Path, current_file: Path, href: str) -> Path | None:
    parsed = urlparse(href)
    if parsed.scheme or href.startswith("#"):
        return None
    target = (current_file.parent / parsed.path).resolve()
    if target.is_dir():
        target = target / "index.md"
    try:
        target.relative_to(root.resolve())
    except ValueError:
        return None
    return target if target.suffix.lower() == ".md" else None

def read_okf_page(bundle_root: str | Path, relative_path: str) -> OKFPage:
    root = Path(bundle_root).resolve()
    page_path = (root / relative_path).resolve()
    try:
        relative = page_path.relative_to(root).as_posix()
    except ValueError as exc:
        raise ValueError("Path escapes bundle root") from exc
    if not page_path.exists():
        raise FileNotFoundError(relative)

    metadata, body = split_frontmatter(page_path.read_text(encoding="utf-8"))
    links = []
    for label, href in MARKDOWN_LINK_RE.findall(body):
        target = safe_resolve(root, page_path, href)
        if target and target.exists():
            links.append(PageLink(
                label=label,
                href=href,
                target_path=target.relative_to(root).as_posix(),
            ))
    return OKFPage(path=relative, metadata=metadata, body=body, links=links)


def page_for_prompt(page: OKFPage, max_chars: int = 16_000) -> str:
    link_rows = [
        {"number": i, "label": link.label, "target_path": link.target_path}
        for i, link in enumerate(page.links)
    ]
    return (
        f"PATH: {page.path}\nMETADATA: {json.dumps(page.metadata, ensure_ascii=False)}\n"
        f"BODY:\n{page.body[:max_chars]}\nLINKS:\n{json.dumps(link_rows, ensure_ascii=False)}"
    )


In [76]:
from pydantic import BaseModel, Field
from typing import Annotated, Literal, TypedDict
import operator
from langgraph.graph import END, START, StateGraph

class NavigationDecision(BaseModel):
    action: Literal["follow", "answer", "abstain"]
    selected_link_number: int | None = None
    evidence: str = ""
    reason: str


class SimpleRetrievalState(TypedDict, total=False):
    question: str
    bundle_root: str
    current_path: str
    current_page: dict
    visited_paths: list[str]
    evidence: Annotated[list[dict], operator.add]
    navigation_trace: Annotated[list[dict], operator.add]
    iteration_count: int
    max_iterations: int
    decision: dict
    answer: str


def content_evidence(page: OKFPage, max_chars: int = 8_000) -> list[dict]:
    # index.md files are routing menus. Other Markdown files contain source material.
    if Path(page.path).name == "index.md":
        return []
    return [{"path": page.path, "content": page.body[:max_chars]}]


def simple_initialize_node(state: SimpleRetrievalState) -> dict:
    return {
        "current_path": "index.md",
        "visited_paths": [],
        "evidence": [],
        "navigation_trace": [],
        "iteration_count": 0,
    }


def simple_read_page_node(state: SimpleRetrievalState) -> dict:
    page = read_okf_page(state["bundle_root"], state["current_path"])
    return {
        "current_page": page.model_dump(),
        "visited_paths": [*state.get("visited_paths", []), page.path],
        "evidence": content_evidence(page),
        "iteration_count": state.get("iteration_count", 0) + 1,
        "navigation_trace": [{"event": "read", "path": page.path, "links": len(page.links)}],
    }


def simple_decide_node(state: SimpleRetrievalState) -> dict:
    page = OKFPage.model_validate(state["current_page"])
    remaining = state["max_iterations"] - state["iteration_count"]
    prompt = f"""
    You are navigating an OKF wiki without vector search. Match by meaning, intent, topic,
    and useful examples; the question does not need to appear verbatim in a page. On this
    step choose exactly one action:
    - follow: select one listed link number that best advances the question
    - answer: the content read so far can support a useful full or partial answer
    - abstain: only when no link is even plausibly relevant and no useful content was read
    Index pages are menus, so normally follow their most relevant link. On a post index,
    prefer Overview or a relevant Discussion theme over Raw comments. Related personal
    experiences and examples are valid evidence for a broad recommendation question.
    Never invent a link. There is no backtracking. Remaining read budget: {remaining}.

    QUESTION: {state['question']}
    VISITED: {state.get('visited_paths', [])}
    CURRENT PAGE:\n{page_for_prompt(page)}
    """
    decision = llm.with_structured_output(NavigationDecision).invoke(prompt)

    valid_follow = (
        decision.action == "follow"
        and decision.selected_link_number is not None
        and 0 <= decision.selected_link_number < len(page.links)
        and page.links[decision.selected_link_number].target_path not in state.get("visited_paths", [])
    )
    if decision.action == "follow" and not valid_follow:
        decision = NavigationDecision(action="abstain", reason="Model selected an invalid link number")
    if decision.action == "follow" and state["iteration_count"] >= state["max_iterations"]:
        decision = NavigationDecision(action="abstain", reason="Maximum iterations reached")

    update = {
        "decision": decision.model_dump(),
        "navigation_trace": [{"event": "decide", **decision.model_dump()}],
    }
    if decision.action == "follow":
        update["current_path"] = page.links[decision.selected_link_number].target_path
    return update


def route_simple_decision(state: SimpleRetrievalState) -> Literal["follow", "answer", "abstain"]:
    # Preserve retrieved source content even if the navigator cannot establish a full answer.
    if state["decision"]["action"] == "abstain" and state.get("evidence"):
        return "answer"
    return state["decision"]["action"]


def simple_answer_node(state: SimpleRetrievalState) -> dict:
    prompt = f"""
    Answer the question from the retrieved source content below. Synthesize relevant
    experiences and implications; do not require the source to repeat the question verbatim.
    Cite supporting OKF paths inline in square brackets, e.g. [posts/123/overview.md].
    If evidence is incomplete, still give the supported partial answer and state its limitation.

    QUESTION: {state['question']}
    EVIDENCE: {json.dumps(state.get('evidence', []), ensure_ascii=False)}
    """
    response = llm.invoke(prompt)
    return {"answer": response.text}


def simple_abstain_node(state: SimpleRetrievalState) -> dict:
    reason = state.get("decision", {}).get("reason", "Insufficient evidence")
    return {"answer": f"ไม่สามารถตอบจากหลักฐานที่ค้นพบภายใน budget นี้: {reason}"}


In [77]:
simple_builder = StateGraph(SimpleRetrievalState)
simple_builder.add_node("initialize", simple_initialize_node)
simple_builder.add_node("read_page", simple_read_page_node)
simple_builder.add_node("decide", simple_decide_node)
simple_builder.add_node("answer", simple_answer_node)
simple_builder.add_node("abstain", simple_abstain_node)

simple_builder.add_edge(START, "initialize")
simple_builder.add_edge("initialize", "read_page")
simple_builder.add_edge("read_page", "decide")
simple_builder.add_conditional_edges(
    "decide",
    route_simple_decision,
    {"follow": "read_page", "answer": "answer", "abstain": "abstain"},
)
simple_builder.add_edge("answer", END)
simple_builder.add_edge("abstain", END)
simple_retrieval_graph = simple_builder.compile()

In [70]:
import json

query = "เงินซื้อความสุขได้จริงหรือไม่"

simple_result = simple_retrieval_graph.invoke(
    {
        "question": query,
        "bundle_root": root,
        "max_iterations": 8,
    },
    {"recursion_limit": 30},
)

In [71]:
from IPython.display import Markdown, display

Markdown(simple_result["answer"])

จากข้อมูลที่ระบุไว้ เงินสามารถช่วยซื้อความสะดวกสบายและช่วยลดความทุกข์ได้จริง แต่มุมมองส่วนใหญ่ชี้ว่าความสุขที่ยั่งยืนไม่ได้มาจากวัตถุหรือตัวเงินเพียงอย่างเดียว [posts/38597354/overview.md] 

ตัวอย่างประสบการณ์จากผู้ที่ประสบความสำเร็จในชีวิตตั้งแต่อายุยังน้อยและมีพร้อมทั้งบ้าน รถ รวมถึงอิสรภาพทางการเงิน สะท้อนให้เห็นว่าความพร้อมด้านวัตถุเหล่านี้อาจยังทำให้รู้สึกเคว้งคว้างและไร้ความหมายได้ [posts/38597354/overview.md] โดยนัยสำคัญแล้ว ความสุขที่แท้จริงและยั่งยืนเกิดขึ้นจากการปรับมุมมองทางใจ การดำเนินชีวิตด้วยความพอดี การฝึกสติ และการเป็นผู้ให้แก่สังคม [posts/38597354/overview.md]

In [73]:
import pandas as pd
pd.DataFrame(simple_result["navigation_trace"])

,event,path,links,action,selected_link_number,evidence,reason
0,read,index.md,9.0,NaN,NaN,NaN,NaN
1,decide,NaN,NaN,follow,3.0,"Lifestyle and Well-being — Personal happiness,...",The question asks whether money can buy true h...
2,read,topics/lifestyle-and-well-being/index.md,2.0,NaN,NaN,NaN,NaN
3,decide,NaN,NaN,follow,0.0,Happiness and Mental Health - Coping with stre...,The question asks whether money can buy true h...
4,read,topics/lifestyle-and-well-being/happiness-and-...,8.0,NaN,NaN,NaN,NaN
5,decide,NaN,NaN,follow,0.0,ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงห...,The question asks whether money can actually b...
6,read,posts/38597354/index.md,4.0,NaN,NaN,NaN,NaN
7,decide,NaN,NaN,follow,1.0,Overview link on a post about whether money ca...,The overview page provides a synthesized summa...
8,read,posts/38597354/overview.md,0.0,NaN,NaN,NaN,NaN
9,decide,NaN,NaN,answer,NaN,บทสนทนานี้อภิปรายเกี่ยวกับประเด็นความสุขที่แท้...,The current overview page directly addresses t...


## Pipeline 2 — Iterative Retrieval with Sequential Subagents

เพิ่ม supervisor แตกคำถามเป็น search tasks แล้วเรียก retrieval subgraph ซ้ำ **ทีละ task** 

```mermaid
flowchart TB
  A["Supervisor plans tasks"] --> B["Run one retrieval subgraph"]
  B --> C{"More tasks?"}
  C -->|yes, sequentially| B
  C -->|no| D["Synthesize answer"]
```


In [78]:
class SearchTask(BaseModel):
    task_id: str
    sub_question: str
    start_link_number: int = Field(description="Index of one link from the root index")


class SearchPlan(BaseModel):
    tasks: list[SearchTask]


class SubagentResult(BaseModel):
    task_id: str
    sub_question: str
    answer: str
    cited_paths: list[str]
    trace: list[dict]


class SubagentState(TypedDict, total=False):
    task_id: str
    sub_question: str
    bundle_root: str
    start_path: str
    current_path: str
    current_page: dict
    visited_paths: list[str]
    evidence: Annotated[list[dict], operator.add]
    trace: Annotated[list[dict], operator.add]
    iteration_count: int
    max_iterations: int
    decision: dict
    result: dict


def subagent_initialize_node(state: SubagentState) -> dict:
    return {
        "current_path": state["start_path"],
        "visited_paths": [],
        "evidence": [],
        "trace": [],
        "iteration_count": 0,
    }


def subagent_read_node(state: SubagentState) -> dict:
    page = read_okf_page(state["bundle_root"], state["current_path"])
    return {
        "current_page": page.model_dump(),
        "visited_paths": [*state.get("visited_paths", []), page.path],
        "evidence": content_evidence(page),
        "iteration_count": state.get("iteration_count", 0) + 1,
        "trace": [{"event": "read", "path": page.path}],
    }


def subagent_decide_node(state: SubagentState) -> dict:
    page = OKFPage.model_validate(state["current_page"])
    prompt = f"""
    Navigate one forward-only path through this OKF wiki for the SUB-QUESTION. Match by
    meaning and intent, not exact wording. Choose follow, answer, or abstain. For follow,
    select exactly one listed unvisited link. Index pages are menus; on a post index prefer
    Overview or a relevant Discussion theme. Answer when content can support a useful full
    or partial response. Abstain only if no link is plausibly relevant and no content helps.
    Do not backtrack or invent paths.

    SUB-QUESTION: {state['sub_question']}
    READS: {state['iteration_count']} / {state['max_iterations']}
    VISITED: {state.get('visited_paths', [])}
    PAGE:\n{page_for_prompt(page)}
    """
    decision = llm.with_structured_output(NavigationDecision).invoke(prompt)
    valid = (
        decision.action != "follow"
        or (
            decision.selected_link_number is not None
            and 0 <= decision.selected_link_number < len(page.links)
            and page.links[decision.selected_link_number].target_path not in state.get("visited_paths", [])
        )
    )
    if not valid:
        decision = NavigationDecision(action="abstain", reason="Invalid or already visited forward link")
    if decision.action == "follow" and state["iteration_count"] >= state["max_iterations"]:
        decision = NavigationDecision(action="abstain", reason="Subagent read budget exhausted")

    update = {
        "decision": decision.model_dump(),
        "trace": [{"event": "decide", **decision.model_dump()}],
    }
    if decision.action == "follow":
        update["current_path"] = page.links[decision.selected_link_number].target_path
    return update


def route_subagent(state: SubagentState) -> Literal["follow", "finish"]:
    return "follow" if state["decision"]["action"] == "follow" else "finish"


def subagent_finish_node(state: SubagentState) -> dict:
    action = state["decision"]["action"]
    if action == "abstain" and not state.get("evidence"):
        answer = f"No sufficient evidence: {state['decision']['reason']}"
    else:
        prompt = f"""
        Answer the sub-question concisely from the retrieved source content. Match by meaning,
        synthesize supported implications, and cite only paths shown in evidence. If support is
        partial, provide the supported part and clearly state the limitation.
        SUB-QUESTION: {state['sub_question']}
        EVIDENCE: {json.dumps(state.get('evidence', []), ensure_ascii=False)}
        """
        answer = llm.invoke(prompt).text
    result = SubagentResult(
        task_id=state["task_id"],
        sub_question=state["sub_question"],
        answer=answer,
        cited_paths=sorted({item["path"] for item in state.get("evidence", [])}),
        trace=state.get("trace", []),
    )
    return {"result": result.model_dump()}


sub_builder = StateGraph(SubagentState)
sub_builder.add_node("initialize", subagent_initialize_node)
sub_builder.add_node("read", subagent_read_node)
sub_builder.add_node("decide", subagent_decide_node)
sub_builder.add_node("finish", subagent_finish_node)
sub_builder.add_edge(START, "initialize")
sub_builder.add_edge("initialize", "read")
sub_builder.add_edge("read", "decide")
sub_builder.add_conditional_edges("decide", route_subagent, {"follow": "read", "finish": "finish"})
sub_builder.add_edge("finish", END)
retrieval_subgraph = sub_builder.compile()


In [79]:
class SupervisorState(TypedDict, total=False):
    question: str
    bundle_root: str
    max_subagents: int
    max_iterations_per_subagent: int
    search_tasks: list[dict]
    current_task_index: int
    subagent_results: Annotated[list[dict], operator.add]
    answer: str


def plan_search_node(state: SupervisorState) -> dict:
    root_page = read_okf_page(state["bundle_root"], "index.md")
    prompt = f"""
    Decompose the QUESTION into at most {state['max_subagents']} complementary search tasks.
    Each task must start from one link number in the root index. Create meaningfully different
    sub-questions (for example: first-person stories, later-career changes, and lessons learned)
    so agents retrieve different evidence. Tasks may share the same root branch when that is
    the best category; diversify evidence, not categories. Do not invent links. Tasks execute
    sequentially, not in parallel.

    QUESTION: {state['question']}
    ROOT INDEX:\n{page_for_prompt(root_page)}
    """
    plan = llm.with_structured_output(SearchPlan).invoke(prompt)
    valid_tasks = []
    for number, task in enumerate(plan.tasks[:state["max_subagents"]], start=1):
        if 0 <= task.start_link_number < len(root_page.links):
            valid_tasks.append({
                "task_id": task.task_id or f"task-{number}",
                "sub_question": task.sub_question,
                "start_path": root_page.links[task.start_link_number].target_path,
            })
    if not valid_tasks:
        valid_tasks = [{
            "task_id": "task-1",
            "sub_question": state["question"],
            "start_path": "index.md",
        }]
    return {"search_tasks": valid_tasks, "current_task_index": 0, "subagent_results": []}


def run_next_subagent_node(state: SupervisorState) -> dict:
    task = state["search_tasks"][state["current_task_index"]]
    # Plain synchronous invoke: one subagent finishes before the next starts.
    result = retrieval_subgraph.invoke(
        {
            **task,
            "bundle_root": state["bundle_root"],
            "max_iterations": state["max_iterations_per_subagent"],
        },
        {"recursion_limit": state["max_iterations_per_subagent"] * 3 + 10},
    )
    return {
        "subagent_results": [result["result"]],
        "current_task_index": state["current_task_index"] + 1,
    }


def route_supervisor(state: SupervisorState) -> Literal["more", "synthesize"]:
    return "more" if state["current_task_index"] < len(state["search_tasks"]) else "synthesize"


def synthesize_node(state: SupervisorState) -> dict:
    prompt = f"""
    Synthesize a grounded answer from sequential subagent reports. Cite only paths in
    cited_paths, preserve disagreements, and say what could not be established.

    QUESTION: {state['question']}
    REPORTS: {json.dumps(state['subagent_results'], ensure_ascii=False)}
    """
    return {"answer": llm.invoke(prompt).text}


supervisor_builder = StateGraph(SupervisorState)
supervisor_builder.add_node("plan_search", plan_search_node)
supervisor_builder.add_node("run_next_subagent", run_next_subagent_node)
supervisor_builder.add_node("synthesize", synthesize_node)
supervisor_builder.add_edge(START, "plan_search")
supervisor_builder.add_edge("plan_search", "run_next_subagent")
supervisor_builder.add_conditional_edges(
    "run_next_subagent",
    route_supervisor,
    {"more": "run_next_subagent", "synthesize": "synthesize"},
)
supervisor_builder.add_edge("synthesize", END)
sequential_subagent_graph = supervisor_builder.compile()


In [80]:
query = "เงินซื้อความสุขได้จริงหรือไม่"

multi_result = sequential_subagent_graph.invoke(
    {
        "question": query,
        "bundle_root": root,
        "max_subagents": 3,
        "max_iterations_per_subagent": 6,
    },
    {"recursion_limit": 50},
)

In [81]:
Markdown(multi_result["answer"])

จากการสังเคราะห์ข้อมูลจากรายงานย่อย สามารถสรุปคำตอบเกี่ยวกับคำถาม **"เงินซื้อความสุขได้จริงหรือไม่"** ได้ดังนี้

* **มุมมองด้านไลฟ์สไตล์และความสุขทางใจ:** มีความเห็นที่สอดคล้องกันว่า **เงินสามารถซื้อความสะดวกสบายและช่วยลดความทุกข์ได้** แต่ความสุขที่แท้จริงและยั่งยืนไม่ได้มาจากวัตถุ โดยความสุขทางใจจะขึ้นอยู่กับการปรับมุมมอง การใช้ชีวิตด้วยความพอดี การฝึกสติ และการเป็นผู้ให้แก่สังคม (`posts/38597354/overview.md`)
* **มุมมองด้านการเงินและการสร้างความมั่งคั่งส่วนบุคคล:** หลักฐานที่มีอยู่มีข้อจำกัด โดยมีตัวอย่างกรณีศึกษาที่สะท้อนว่า **การมีรายได้ดีแต่ใช้ชีวิตประหยัดสุดขั้วและไม่มีความสุข** (เช่น การเปรียบเทียบตนเองกับผู้อื่นจนรู้สึกว่าตนเองยากจนและล้มเหลว) สามารถเป็นปัญหาที่เกิดขึ้นได้เช่นกัน (`posts/40980378/overview.md`)

**ประเด็นที่ไม่สามารถระบุได้:** 
* ข้อมูลที่มีอยู่ไม่เพียงพอที่จะระบุถึงผลกระทบของการเงินที่มีต่อคุณภาพชีวิตในครอบครัวและสังคมในภาพรวมได้ เนื่องจากขาดหลักฐานและข้อมูลที่เกี่ยวข้องในส่วนนี้

In [82]:
pd.DataFrame([
    {
        "task_id": item["task_id"],
        "sub_question": item["sub_question"],
        "cited_paths": ", ".join(item["cited_paths"]),
        "steps": len(item["trace"]),
    }
    for item in multi_result["subagent_results"]
])

,task_id,sub_question,cited_paths,steps
0,task_1,เงินและทรัพย์สินมีความสัมพันธ์อย่างไรกับความสุ...,posts/40980378/overview.md,8
1,task_2,มุมมองด้านไลฟ์สไตล์และความสุขทางใจระบุไว้อย่าง...,posts/38597354/overview.md,8
2,task_3,การเงินส่งผลต่อคุณภาพชีวิตในครอบครัวและสังคมใน...,,4


## Comparison

| Dimension | Simple iterative | Sequential subagents |
|---|---|---|
| Search breadth | หนึ่งเส้นทาง | หลายเส้นทางที่ supervisor วางแผน |
| Execution | agent เดียว | subgraph หลาย invocation แบบทีละงาน |
| Backtracking | ไม่มี | ไม่มี |
| Parallel / Send API | ไม่ใช้ | ไม่ใช้ |
| Cost/latency | ต่ำกว่า | สูงกว่าตามจำนวน tasks |
| Failure mode | เลือกลิงก์ผิดแล้วอาจพลาดคำตอบ | task decomposition หรือ branch เริ่มต้นอาจผิด |
| Explainability | trace เดียวอ่านง่าย | trace แยกตาม task และรวม citations ตอนท้าย |

**อภิปราย:** sequential subagents เพิ่ม coverage แต่ไม่ได้แก้ปัญหา navigation error ภายใน branch เพราะไม่ใช้ backtracking การออกแบบ index และ link descriptions จึงมีผลต่อ retrieval quality โดยตรง
